# Modelo 7d — Score de expansão

Estima a atratividade de cada município (dentre SP, RJ, DF, PR) para expansão do
EV ChargeOps, combinando frota BEV/PHEV (RENAVAM), cobertura de eletropostos
públicos (Open Charge Map) e população (IBGE) -- os 3 indicadores já carregados
em `dim_geography` na Etapa 6.5.

## Mudança de abordagem em relação ao documento original

O documento da Sprint 01 descreve este modelo como "regressão logística". Isso é,
por definição, um método **supervisionado** -- exige um target binário conhecido
(ex: "essa expansão deu certo? sim/não") para treinar. Não temos nenhum
município com resultado real de expansão: é tudo hipotético neste estágio do
projeto.

Em vez de fabricar um target sintético artificial (o que criaria uma falsa
sensação de "aprendizado" a partir de uma regra que nós mesmos definimos),
optamos por um **índice composto / score ponderado**, não-supervisionado --
mais honesto sobre o que o método realmente faz: combina indicadores conhecidos
em um único número comparável, sem pretender ter "aprendido" um padrão de
sucesso que não existe nos dados.

O uso pretendido (mapa colorido, verde = maior atratividade, vermelho = menor)
é melhor servido por um score contínuo do que por uma classificação binária de
qualquer forma.

## Interpretação de cada indicador (decisão de negócio)

- **Frota BEV/PHEV alta → bom sinal.** Indica demanda real de recarga já existente.
- **População alta → bom sinal.** Mais moradores, mais condomínios/prédios, mais
  escala potencial para o produto.
- **Cobertura de eletropostos públicos alta → reduz o score.** O EV ChargeOps
  não compete com rede pública de recarga rápida -- opera em condomínios e
  prédios privados. Cobertura pública alta é um proxy de que a demanda de
  recarga já está sendo atendida por infraestrutura pública, reduzindo a
  lacuna que o produto preenche. **Importante: usamos cobertura RELATIVA à
  frota** (eletropostos por 1.000 veículos elétricos), não o valor absoluto
  -- cobertura absoluta mistura "tamanho da cidade" com "densidade de
  infraestrutura", penalizando injustamente cidades grandes que naturalmente
  têm mais eletropostos e mais frota ao mesmo tempo (ver seção de calibração
  abaixo para o caso real que motivou essa correção).

## Fórmula

Cada indicador é normalizado (Min-Max, 0 a 1) antes de combinar -- sem isso,
população (escala de milhões) dominaria frota (escala de milhares) na soma.

```
score = peso_frota * frota_norm + peso_populacao * populacao_norm
        - peso_cobertura * cobertura_norm
```

Pesos: frota e população pesam igualmente como indicadores de mercado potencial;
cobertura pesa como fator de desconto.


## Setup

In [1]:
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import psycopg2
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", None)

In [2]:
DB_CONFIG = {
    "host": os.getenv("POSTGRES_HOST", "localhost"),
    "port": os.getenv("POSTGRES_PORT", "5432"),
    "dbname": os.getenv("POSTGRES_DB", "evchargeops"),
    "user": os.getenv("POSTGRES_USER", "evchargeops"),
    "password": os.getenv("POSTGRES_PASSWORD", ""),
}

MODELS_DIR = Path("output")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Pesos do score composto (decisão de negócio, documentada acima).
PESO_FROTA = 0.4
PESO_POPULACAO = 0.4
PESO_COBERTURA = 0.2  # subtraído, não somado

## Carga dos dados

In [3]:
def fetch_geography(conn) -> pd.DataFrame:
    query = """
        SELECT
            geo_id, city, state, ibge_municipio_id,
            population_estimate, ev_fleet_bev_count, ev_fleet_phev_count,
            chargepoints_count
        FROM dim_geography
    """
    return pd.read_sql(query, conn)

In [4]:
conn = psycopg2.connect(**DB_CONFIG)
try:
    geo_df = fetch_geography(conn)
finally:
    conn.close()

geo_df["frota_total"] = geo_df["ev_fleet_bev_count"] + geo_df["ev_fleet_phev_count"]

print(f"{len(geo_df)} municípios carregados.")
geo_df.describe()

1137 municípios carregados.


/tmp/ipykernel_179172/1260122234.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,ibge_municipio_id,population_estimate,ev_fleet_bev_count,ev_fleet_phev_count,chargepoints_count,frota_total
count,1.137000e+03,1.137000e+03,1137.000000,1137.000000,1137.000000,1137.000000
mean,3.717553e+06,6.877112e+04,85.521548,74.457344,100.044855,159.978892
std,3.031043e+05,4.315195e+05,831.089747,785.783065,28.412869,1604.845944
min,3.300100e+06,9.320000e+02,0.000000,0.000000,13.000000,0.000000
25%,3.516853e+06,5.693000e+03,0.000000,0.000000,74.000000,1.000000
50%,3.542503e+06,1.286900e+04,2.000000,2.000000,124.000000,4.000000
75%,4.108007e+06,3.452100e+04,12.000000,10.000000,124.000000,21.000000
max,5.300108e+06,1.190496e+07,21045.000000,18774.000000,124.000000,37548.000000


## Tratamento de nulos

`population_estimate` pode ter nulos pontuais (falha de API já documentada na
Etapa 6.5). Municípios sem população resolvida são excluídos do score -- não
dá para comparar atratividade sem esse dado.

In [5]:
antes = len(geo_df)
geo_df = geo_df.dropna(subset=["population_estimate"]).copy()
print(f"Municípios removidos por falta de população: {antes - len(geo_df)}")
print(f"Municípios restantes: {len(geo_df)}")

Municípios removidos por falta de população: 0
Municípios restantes: 1137


## Normalização (Min-Max) e cálculo do score

In [6]:
# Cobertura é usada em termos RELATIVOS à frota (eletropostos por 1000
# veículos elétricos), não em valor absoluto. Motivo: cobertura absoluta
# mistura "tamanho da cidade" com "densidade de infraestrutura" -- uma
# cidade grande naturalmente tem mais eletropostos E mais frota, então usar
# o número absoluto penaliza injustamente cidades grandes (foi o que
# aconteceu com Curitiba na primeira versão: cobertura absoluta alta
# afundou o score, mesmo com frota e população relevantes). A métrica
# relativa mede o que de fato importa para o negócio: quanto de
# infraestrutura pública já existe para cada veículo elétrico da região.
# Piso mínimo de frota (30 veículos) para entrar no cálculo relativo --
# sem isso, um município com 2 veículos e 1 eletroposto teria uma proporção
# de 500 por 1000 EVs, artificialmente extrema por causa do denominador
# pequeno, não porque a cobertura ali seja de fato desproporcional.
# Municípios abaixo do piso usam a mediana da cobertura relativa dos
# demais, evitando tanto o ruído do denominador pequeno quanto a exclusão
# do município da análise.
FROTA_MINIMA_PARA_RELATIVO = 30

cobertura_relativa_bruta = np.where(
    geo_df["frota_total"] > 0,
    (geo_df["chargepoints_count"] / geo_df["frota_total"]) * 1000,
    np.nan,
)
mediana_cobertura = np.nanmedian(
    np.where(geo_df["frota_total"] >= FROTA_MINIMA_PARA_RELATIVO, cobertura_relativa_bruta, np.nan)
)
geo_df["cobertura_por_1000_evs"] = np.where(
    geo_df["frota_total"] >= FROTA_MINIMA_PARA_RELATIVO,
    cobertura_relativa_bruta,
    mediana_cobertura,
)

scaler = MinMaxScaler()
norm_cols = ["frota_total", "population_estimate", "cobertura_por_1000_evs"]
normalized = scaler.fit_transform(geo_df[norm_cols])

geo_df["frota_norm"] = normalized[:, 0]
geo_df["populacao_norm"] = normalized[:, 1]
geo_df["cobertura_norm"] = normalized[:, 2]

geo_df["expansion_score"] = (
    PESO_FROTA * geo_df["frota_norm"]
    + PESO_POPULACAO * geo_df["populacao_norm"]
    - PESO_COBERTURA * geo_df["cobertura_norm"]
)

# Reescala o score final para 0-100 -- mais legível para o mapa do Streamlit
# (verde = alto, vermelho = baixo) do que um número decimal entre -0.2 e 0.8.
score_scaler = MinMaxScaler(feature_range=(0, 100))
geo_df["expansion_score_0_100"] = score_scaler.fit_transform(geo_df[["expansion_score"]])

geo_df[["city", "state", "frota_total", "population_estimate", "cobertura_por_1000_evs", "expansion_score_0_100"]].sort_values(
    "expansion_score_0_100", ascending=False
).head(15)

,city,state,frota_total,population_estimate,cobertura_por_1000_evs,expansion_score_0_100
562,São Paulo,SP,33634,11904961,3.686746,100.000000
737,Brasília,DF,37548,2996899,0.346224,73.094656
712,Rio de Janeiro,RJ,13848,6730729,3.321779,59.807033
831,Curitiba,PR,9728,1830795,7.606908,37.993463
108,Campinas,SP,7159,1187974,17.320855,32.827297
212,Guarulhos,SP,1380,1349100,89.855072,26.592961
487,Ribeirão Preto,SP,2837,731639,43.708142,26.280156
581,Sorocaba,SP,2147,762172,57.755007,25.548213
544,São Bernardo do Campo,SP,1644,841154,75.425791,25.176238
534,Santo André,SP,1514,782048,81.902246,24.791227


## Top 15 municípios por score de expansão

In [7]:
top15 = geo_df.sort_values("expansion_score_0_100", ascending=False).head(15)
top15[["city", "state", "frota_total", "population_estimate", "cobertura_por_1000_evs", "expansion_score_0_100"]]

,city,state,frota_total,population_estimate,cobertura_por_1000_evs,expansion_score_0_100
562,São Paulo,SP,33634,11904961,3.686746,100.000000
737,Brasília,DF,37548,2996899,0.346224,73.094656
712,Rio de Janeiro,RJ,13848,6730729,3.321779,59.807033
831,Curitiba,PR,9728,1830795,7.606908,37.993463
108,Campinas,SP,7159,1187974,17.320855,32.827297
212,Guarulhos,SP,1380,1349100,89.855072,26.592961
487,Ribeirão Preto,SP,2837,731639,43.708142,26.280156
581,Sorocaba,SP,2147,762172,57.755007,25.548213
544,São Bernardo do Campo,SP,1644,841154,75.425791,25.176238
534,Santo André,SP,1514,782048,81.902246,24.791227


## Bottom 15 (menor atratividade)

In [8]:
bottom15 = geo_df.sort_values("expansion_score_0_100", ascending=True).head(15)
bottom15[["city", "state", "frota_total", "population_estimate", "cobertura_por_1000_evs", "expansion_score_0_100"]]

,city,state,frota_total,population_estimate,cobertura_por_1000_evs,expansion_score_0_100
432,Piracaia,SP,30,26795,4133.333333,0.000000
575,Serra Negra,SP,30,31047,4133.333333,0.014930
348,Mongaguá,SP,30,64845,4133.333333,0.133607
260,Itápolis,SP,31,40445,4000.000000,0.723282
138,Cordeirópolis,SP,32,25286,3875.000000,1.303265
290,José Bonifácio,SP,32,37992,3875.000000,1.347881
608,Tietê,SP,32,38723,3875.000000,1.350448
148,Cruzeiro,SP,32,76444,3875.000000,1.482900
0,Adamantina,SP,33,35673,3757.575758,1.934641
574,Serrana,SP,34,45580,3647.058824,2.529402


## Sanity check

Confere se os municípios do piloto (com pontos reais do EV ChargeOps) fazem
sentido dentro da distribuição de score -- não é uma validação estatística
formal (não há gabarito), mas uma checagem de plausibilidade.

**Nota de calibração**: a métrica de cobertura relativa (`cobertura_por_1000_evs`)
pode ainda produzir scores baixos para municípios com proporção alta de
eletropostos por veículo -- isso é o comportamento correto do modelo, não um
bug. Um município com muitos eletropostos por veículo já tem sua demanda de
recarga bem atendida pela rede pública, reduzindo a lacuna de mercado para o
EV ChargeOps. Vale conferir, ao rodar com dados reais completos (todos os
municípios de SP/RJ/DF/PR), se essa métrica não está sendo distorcida por
municípios com poucos veículos e um único eletroposto isolado (denominador
pequeno gerando proporção artificialmente alta) -- nesse caso, considerar um
piso mínimo de frota para entrar no cálculo relativo.

In [9]:
# Municípios onde o piloto está (dim_points.location -> ver Etapa 6.5)
piloto_cities = ["São Paulo", "Rio de Janeiro"]
piloto_scores = geo_df[geo_df["city"].isin(piloto_cities)][
    ["city", "state", "frota_total", "population_estimate", "cobertura_por_1000_evs", "expansion_score_0_100"]
]
print("Score dos municípios do piloto:")
piloto_scores

Score dos municípios do piloto:


,city,state,frota_total,population_estimate,cobertura_por_1000_evs,expansion_score_0_100
562,São Paulo,SP,33634,11904961,3.686746,100.000000
712,Rio de Janeiro,RJ,13848,6730729,3.321779,59.807033


## Persistência\n\nSalva a tabela completa de scores (usada para colorir o mapa no Streamlit) e os parâmetros de normalização (para aplicar o mesmo cálculo a municípios novos sem reprocessar tudo).

In [10]:
expansion_model = {
    "scaler_features": scaler,
    "scaler_score": score_scaler,
    "norm_cols": norm_cols,
    "weights": {"frota": PESO_FROTA, "populacao": PESO_POPULACAO, "cobertura": PESO_COBERTURA},
}

joblib.dump(expansion_model, MODELS_DIR / "score_expansao_modelo.joblib")
geo_df.to_csv(MODELS_DIR / "score_expansao_municipios.csv", index=False)

print("Artefatos salvos em:", MODELS_DIR.resolve())
print(" -", "score_expansao_modelo.joblib")
print(" -", "score_expansao_municipios.csv")

Artefatos salvos em: /home/vinivaliati/projects/computer_science_fiap_challenge_2026/models/output
 - score_expansao_modelo.joblib
 - score_expansao_municipios.csv
